In [1]:
!pip install seaborn
!pip install torch
!pip install torchvision
!pip install scikit-learn

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import time
import os
import sys

In [3]:
import os

import tensorflow as tf
from tensorflow import keras

print(tf.version.VERSION)

2.4.1


In [4]:
TRAIN_DIR = "./face_extraction/faceforensis_face_extract/train/"
VAL_DIR = "./face_extraction/faceforensis_face_extract/val/"
TEST_DIR = "./face_extraction/faceforensis_face_extract/test/"

transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load datasets
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=transform)

In [5]:
class_names = train_dataset.classes
num_classes = len(class_names)

In [6]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
import os

# --- CONFIGURAÇÕES ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- DEFINIR A CLASSE DO MODELO ---
class Model(nn.Module):
    def __init__(self, num_classes):
        super(Model, self).__init__()
        self.inception_model = models.inception_v3(pretrained=True, aux_logits=False)
        self.inception_model.fc = nn.Linear(self.inception_model.fc.in_features, num_classes)

    def forward(self, x):
        return self.inception_model(x)

# --- DEFINIR CAMINHOS E TRANSFORMAÇÕES ---
TEST_DIR = "./face_extraction/faceforensis_face_extract/test/"

transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# --- CARREGAR O MODELO ---
num_classes = len(test_dataset.classes)
class_names = test_dataset.classes

model = Model(num_classes=num_classes).to(device)
model.load_state_dict(torch.load('best_inceptionv3_model_multiclass.h5', map_location=device))
model.eval()


Model(
  (inception_model): Inception3(
    (Conv2d_1a_3x3): BasicConv2d(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    )
    (Conv2d_2a_3x3): BasicConv2d(
      (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    )
    (Conv2d_2b_3x3): BasicConv2d(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    )
    (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (Conv2d_3b_1x1): BasicConv2d(
      (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    )
    (Conv2d_4a_3x3): B

In [ ]:
from sklearn.metrics import classification_report

all_predictions = []
all_actual_values = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.cpu().numpy())
        all_actual_values.extend(targets.cpu().numpy())

# Exibir o classification report
print(classification_report(all_actual_values, all_predictions, target_names=class_names))


In [ ]:
!pip install grad-cam


In [7]:
!pip install captum

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
from PIL import Image
from captum.attr import LayerGradCam, Saliency, IntegratedGradients, InputXGradient
from captum.attr import LayerAttribution
from captum.attr import NoiseTunnel
    
def load_image_tensor(img_path):
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0)
    img_tensor.requires_grad_()  # Adicionado para evitar o warning do Captum
    img_np = np.array(img.resize((299, 299))).astype(np.float32) / 255.0
    return img_tensor, img_np

def normalize_attr(attr):
    attr = np.maximum(attr, 0)
    if np.max(attr) > 0:
        attr = attr / np.max(attr)
    return np.stack([attr] * 3, axis=-1)

def generate_visualizations(model, img_tensor, img_np, label):
    target_layer = model.inception_model.Mixed_7c

    gradcam = LayerGradCam(model, target_layer)
    gradcam_attr = gradcam.attribute(img_tensor, target=label)
    gradcam_attr = LayerAttribution.interpolate(gradcam_attr, (299, 299))
    gradcam_attr = gradcam_attr.squeeze().detach().cpu().numpy()  # <-- .cpu() adicionado aqui

    saliency = Saliency(model)
    saliency_attr = saliency.attribute(img_tensor, target=label)
    saliency_attr = saliency_attr.squeeze().abs().detach().cpu().numpy().mean(axis=0)

    inputxgrad = InputXGradient(model)
    inputxgrad_attr = inputxgrad.attribute(img_tensor, target=label)
    inputxgrad_attr = inputxgrad_attr.squeeze().detach().cpu().numpy().mean(axis=0)

    integrated = IntegratedGradients(model)
    integrated_attr = integrated.attribute(img_tensor, target=label, n_steps=10)
    integrated_attr = integrated_attr.squeeze().detach().cpu().numpy().mean(axis=0)

    smoothgrad = NoiseTunnel(saliency)
    smoothgrad_attr = smoothgrad.attribute(img_tensor, nt_type="smoothgrad", target=label)
    smoothgrad_attr = smoothgrad_attr.squeeze().detach().cpu().numpy().mean(axis=0)

    vis = []
    for attr in [gradcam_attr, saliency_attr, inputxgrad_attr, integrated_attr, smoothgrad_attr]:
        blended = np.clip(img_np * 0.5 + normalize_attr(attr) * 0.5, 0, 1)
        vis.append(blended)
    return vis



# Classes e imagens
image_groups = {
    "ORIGINAL": [
        "./face_extraction/faceforensis_face_extract/train/original/original_000_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/original/original_004_frame2.png",
        "./face_extraction/faceforensis_face_extract/train/original/original_006_frame1.png",
        "./face_extraction/faceforensis_face_extract/train/original/original_011_frame4.png",
        "./face_extraction/faceforensis_face_extract/train/original/original_018_frame3.png",
    ],
    "FACE2FACE": [
        "./face_extraction/faceforensis_face_extract/train/face2face/Face2Face_000_003_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/face2face/Face2Face_002_006_frame3.png",
        "./face_extraction/faceforensis_face_extract/train/face2face/Face2Face_006_002_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/face2face/Face2Face_010_005_frame3.png",
        "./face_extraction/faceforensis_face_extract/train/face2face/Face2Face_017_803_frame3.png",
    ],
    "FACESHIFTER": [
        "./face_extraction/faceforensis_face_extract/train/FaceShifter/FaceShifter_000_003_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/FaceShifter/FaceShifter_002_006_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/FaceShifter/FaceShifter_006_002_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/FaceShifter/FaceShifter_010_005_frame3.png",
        "./face_extraction/faceforensis_face_extract/train/FaceShifter/FaceShifter_017_803_frame3.png",
    ],
    "FACESWAP": [
        "./face_extraction/faceforensis_face_extract/train/FaceSwap/FaceSwap_000_003_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/FaceSwap/FaceSwap_002_006_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/FaceSwap/FaceSwap_006_002_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/FaceSwap/FaceSwap_010_005_frame3.png",
        "./face_extraction/faceforensis_face_extract/train/FaceSwap/FaceSwap_017_803_frame3.png",
    ],
    "DEEPFAKEDETECTION": [
        "./face_extraction/faceforensis_face_extract/train/DeepFakeDetection/DeepFakeDetection_01_02__talking_against_wall__YVGY8LOK_frame4.png",
        "./face_extraction/faceforensis_face_extract/train/DeepFakeDetection/DeepFakeDetection_01_04__talking_angry_couch__0XUW13RW_frame3.png",
        "./face_extraction/faceforensis_face_extract/train/DeepFakeDetection/DeepFakeDetection_01_15__kitchen_still__02HILKYO_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/DeepFakeDetection/DeepFakeDetection_01_21__walk_down_hall_angry__03X7CELV_frame1.png",
        "./face_extraction/faceforensis_face_extract/train/DeepFakeDetection/DeepFakeDetection_01_27__meeting_serious__ZYCZ30C0_frame3.png",
    ],
    "DEEPFAKES": [
        "./face_extraction/faceforensis_face_extract/train/deepfakes/Deepfakes_000_003_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/deepfakes/Deepfakes_002_006_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/deepfakes/Deepfakes_006_002_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/deepfakes/Deepfakes_010_005_frame3.png",
        "./face_extraction/faceforensis_face_extract/train/deepfakes/Deepfakes_017_803_frame3.png",
    ],
    "NEURALTEXTURES": [
        "./face_extraction/faceforensis_face_extract/train/NeuraTextures/NeuralTextures_000_003_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/NeuraTextures/NeuralTextures_002_006_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/NeuraTextures/NeuralTextures_006_002_frame5.png",
        "./face_extraction/faceforensis_face_extract/train/NeuraTextures/NeuralTextures_010_005_frame3.png",
        "./face_extraction/faceforensis_face_extract/train/NeuraTextures/NeuralTextures_017_803_frame3.png",
    ],
}

def show_explanations_grid():
    techniques = ["GradCAM", "Saliency", "InputXGradient", "IntegratedGradients", "SmoothGrad"]
    fig, axs = plt.subplots(len(image_groups), len(techniques), figsize=(5*len(techniques), 4*len(image_groups)))
    for row_idx, (label, image_list) in enumerate(image_groups.items()):
        img_tensor, img_np = load_image_tensor(image_list[0])  # primeira imagem da classe
        img_tensor = img_tensor.to(device)
        pred = torch.argmax(torch.nn.functional.softmax(model(img_tensor), dim=1)).item()
        vis_list = generate_visualizations(model, img_tensor, img_np, pred)
        for col_idx, vis in enumerate(vis_list):
            axs[row_idx, col_idx].imshow(vis)
            axs[row_idx, col_idx].axis('off')
            if row_idx == 0:
                axs[row_idx, col_idx].set_title(techniques[col_idx], fontsize=12)
            if col_idx == 0:
                axs[row_idx, col_idx].set_ylabel(label, fontsize=12)
    plt.tight_layout()
    plt.savefig("explicabilidade_grid_inception.png")  # Salva o resultado como imagem
    print("✅ Figura salva como 'explicabilidade_grid_inception.png'")
    plt.show(block=True)

# Executar
if __name__ == "__main__":
    show_explanations_grid()

In [8]:
pip install pandas scikit-learn matplotlib seaborn openpyxl joblib

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [9]:
!pip install xlrd

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


In [10]:
pip install scikit-image


You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np
from skimage.segmentation import slic
from scipy.stats import entropy
import os
import pandas as pd
import matplotlib.pyplot as plt
from skimage.transform import resize
from captum.attr import IntegratedGradients, Saliency, InputXGradient, NoiseTunnel
from skimage.transform import resize
import gc


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_ROOT = "./face_extraction/faceforensis_face_extract/test/"
class_names = ["DeepFakeDetection", "FaceShifter", "FaceSwap", "NeuraTextures", "deepfakes", "face2face", "original"]
explic_methods = ["Saliency", "InputXGradient", "IntegratedGradients", "SmoothGrad", "GradCAM"]

print("Iniciando.")


def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(DEVICE)
    np_img = np.array(image.resize((299, 299))) / 255.0
    return tensor, np_img

def make_gradcam(model, img_tensor, target_layer='Mixed_7c'):
    gradients, activations = [], []

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0])

    def forward_hook(module, input, output):
        activations.append(output)

    if hasattr(model, 'inception_model'):
        target_module = dict(model.inception_model.named_modules())[target_layer]
    else:
        target_module = dict(model.named_modules())[target_layer]

    fh = target_module.register_forward_hook(forward_hook)
    bh = target_module.register_full_backward_hook(backward_hook)

    img_tensor.requires_grad_(True)
    # NÃO usar torch.no_grad aqui:
    output = model(img_tensor)
    class_idx = torch.argmax(output)
    loss = output[0, class_idx]
    model.zero_grad()
    loss.backward()

    grad = gradients[0]
    act = activations[0]
    pooled_grad = torch.mean(grad, dim=[0, 2, 3])
    for i in range(act.shape[1]):
        act[:, i, :, :] *= pooled_grad[i]
    heatmap = act.mean(dim=1).squeeze().detach().cpu().numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (heatmap.max() + 1e-12)

    fh.remove()
    bh.remove()
    return heatmap

def get_saliency_map(method, model, input_tensor, target_idx):
    input_tensor = input_tensor.clone().detach().requires_grad_(True)

    if method == "Saliency":
        explainer = Saliency(model)
        attributions = explainer.attribute(input_tensor, target=target_idx)
    elif method == "InputXGradient":
        explainer = InputXGradient(model)
        attributions = explainer.attribute(input_tensor, target=target_idx)
    elif method == "IntegratedGradients":
        explainer = IntegratedGradients(model)
        attributions = explainer.attribute(input_tensor, target=target_idx, n_steps=10, internal_batch_size=5)
    elif method == "SmoothGrad":
        saliency = Saliency(model)
        explainer = NoiseTunnel(saliency)
        attributions = explainer.attribute(input_tensor, nt_type='smoothgrad', nt_samples=15, target=target_idx)
    elif method == "GradCAM":
        cam = make_gradcam(model, input_tensor)
        cam_resized = resize(cam, (224, 224), mode='reflect', anti_aliasing=True)
        return cam_resized
    else:
        raise ValueError(f"Método {method} desconhecido.")

    attr_np = attributions.detach().cpu().numpy().squeeze()
    attr_np = np.abs(attr_np).sum(axis=0)
    attr_np = (attr_np - attr_np.min()) / (attr_np.max() - attr_np.min() + 1e-12)
    return attr_np

def salience_entropy(smap):
    smap = smap / (np.sum(smap) + 1e-12)
    return entropy(smap.flatten() + 1e-12)

def reaction_to_noise(model, img_tensor, salmap):
    noisy = img_tensor + torch.normal(0, 0.05, img_tensor.shape).to(DEVICE)
    output = model(noisy)
    target_idx = torch.argmax(output).item()
    noisy_salmap = get_saliency_map("GradCAM", model, noisy, target_idx)
    
    # Redimensiona o mapa ruidoso para o tamanho do original
    noisy_salmap_resized = resize(noisy_salmap, salmap.shape, mode='reflect', anti_aliasing=True)
    
    return np.mean(np.abs(salmap - noisy_salmap_resized))

def geometrical_robustness(model, raw_img, salmap):
    flipped = np.flip(raw_img, axis=1).copy()  # flip horizontal
    flipped_tensor = torch.from_numpy(flipped.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)
    flipped_tensor.requires_grad_(True)

    output = model(flipped_tensor)
    target_idx = torch.argmax(output).item()
    flipped_salmap = get_saliency_map("GradCAM", model, flipped_tensor, target_idx)

    # Redimensiona para o shape original do mapa
    flipped_salmap_resized = resize(flipped_salmap, salmap.shape, mode='reflect', anti_aliasing=True)

    return np.mean(np.abs(salmap - flipped_salmap_resized))

def accuracy_over_segmentation(model, image, saliency_map, segments=50):
    salmap_resized = resize(saliency_map, (image.shape[0], image.shape[1]), mode='reflect', anti_aliasing=True)
    img_uint8 = (image * 255).astype(np.uint8)
    segs = slic(img_uint8, n_segments=segments, compactness=20, enforce_connectivity=True, start_label=0)
    relevance = np.array([np.mean(salmap_resized[segs == i]) for i in range(np.max(segs) + 1)])
    top_k = np.argsort(relevance)[-int(segments * 0.2):]
    mask = np.isin(segs, top_k).astype(np.float32)
    if mask.ndim == 2:
        mask = np.repeat(mask[:, :, np.newaxis], 3, axis=2)
    degraded = image * (1 - mask)
    with torch.no_grad():
        orig_pred = model(torch.from_numpy(image.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE))[0]
        degr_pred = model(torch.from_numpy(degraded.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE))[0]
    return np.abs(orig_pred.cpu().numpy() - degr_pred.cpu().numpy()).mean(), segs

print("Modelo carregado com sucesso.")

image_files = []
image_labels = []
for folder in class_names:
    folder_path = os.path.join(DATASET_ROOT, folder)
    files = sorted([f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.png'))])
    for f in files:
        image_files.append(os.path.join(folder_path, f))
        image_labels.append(folder)


resultados = []
n_rows = len(image_files)
n_cols = len(explic_methods) + 1

# fig, axs = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))

for row, (path, label) in enumerate(zip(image_files, image_labels)):
    img_tensor, raw_img = preprocess_image(path)
    output = model(img_tensor)
    pred_idx = torch.argmax(output, dim=1).item()

    for col, method in enumerate(explic_methods, start=1):
        salmap = get_saliency_map(method, model, img_tensor, pred_idx)
        entropy_val = salience_entropy(salmap)
        noise_val = reaction_to_noise(model, img_tensor, salmap)
        geo_val = geometrical_robustness(model, raw_img, salmap)
        aos_val, _ = accuracy_over_segmentation(model, raw_img, salmap)

        resultado = {
            "imagem": os.path.basename(path),
            "classe_real": label,
            "classe_predita": class_names[pred_idx],
            "metodo_explicabilidade": method,
            "entropia": entropy_val,
            "ruido": noise_val,
            "geo": geo_val,
            "aos": aos_val
        }
        resultados.append(resultado)

        print(f"[{row+1}/{n_rows}] Imagem: {os.path.basename(path)} | Técnica: {method} | Entropia: {entropy_val:.4f} | Ruído: {noise_val:.4f} | Geo: {geo_val:.4f} | AOS: {aos_val:.4f}")

    del img_tensor, raw_img, output
    torch.cuda.empty_cache()
    gc.collect()



#plt.tight_layout()
#plt.savefig("grid_explicabilidade_todas_imagens.png", dpi=300)
#print("Grid salvo como grid_explicabilidade_todas_imagens.png")

df_resultados = pd.DataFrame(resultados)
df_resultados.to_excel("resultados_explicabilidade_todas_imagens_inception.xlsx", index=False)
print("Planilha salva como resultados_explicabilidade_todas_imagens_inception.xlsx")

Iniciando.
Modelo carregado com sucesso.
[1/6999] Imagem: DeepFakeDetection_01_02__meeting_serious__YVGY8LOK_frame1.png | Técnica: Saliency | Entropia: 11.0522 | Ruído: 0.2838 | Geo: 0.2172 | AOS: 2.9338
[1/6999] Imagem: DeepFakeDetection_01_02__meeting_serious__YVGY8LOK_frame1.png | Técnica: InputXGradient | Entropia: 10.9450 | Ruído: 0.2754 | Geo: 0.2248 | AOS: 3.5435
[1/6999] Imagem: DeepFakeDetection_01_02__meeting_serious__YVGY8LOK_frame1.png | Técnica: IntegratedGradients | Entropia: 10.9381 | Ruído: 0.2665 | Geo: 0.2293 | AOS: 3.5116
[1/6999] Imagem: DeepFakeDetection_01_02__meeting_serious__YVGY8LOK_frame1.png | Técnica: SmoothGrad | Entropia: 11.2067 | Ruído: 0.2426 | Geo: 0.1956 | AOS: 3.2202
[1/6999] Imagem: DeepFakeDetection_01_02__meeting_serious__YVGY8LOK_frame1.png | Técnica: GradCAM | Entropia: 10.5795 | Ruído: 0.1169 | Geo: 0.1709 | AOS: 3.5780
[2/6999] Imagem: DeepFakeDetection_01_02__meeting_serious__YVGY8LOK_frame2.png | Técnica: Saliency | Entropia: 11.0305 | Ruído

In [ ]:
print("Nomes dos módulos do modelo:")
for name, module in model.named_modules():
    print(name)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np
from skimage.segmentation import slic
from scipy.stats import entropy
import os
import pandas as pd
import matplotlib.pyplot as plt # Mantido caso você queira visualizar, mas comentado no loop
from skimage.transform import resize
from captum.attr import IntegratedGradients, Saliency, InputXGradient, NoiseTunnel
import gc

# --- DEFINIR A CLASSE DO MODELO INCEPTIONV3 ---
# Esta classe precisa estar definida para que o script saiba como carregar seu modelo
class Model(nn.Module):
    def __init__(self, num_classes):
        super(Model, self).__init__()
        self.inception_model = models.inception_v3(pretrained=False, aux_logits=False) # pretrained=False se você carrega os pesos
        # Substitui a última camada para o número de classes correto
        self.inception_model.fc = nn.Linear(self.inception_model.fc.in_features, num_classes)

    def forward(self, x):
        return self.inception_model(x)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_ROOT = "./face_extraction/faceforensis_face_extract/test/"
# Corrija os nomes das classes para corresponderem aos diretórios e ao que o modelo foi treinado
class_names = ["DeepFakeDetection", "FaceShifter", "FaceSwap", "NeuraTextures", "deepfakes", "face2face", "original"]

explic_methods = ["Saliency", "InputXGradient", "IntegratedGradients", "SmoothGrad", "GradCAM"]

print("Iniciando.")

# --- CARREGAR O MODELO ---
# Você precisa carregar o modelo antes de usá-lo
# Primeiramente, determine o número correto de classes.
# Se `DATASET_ROOT` contiver todos os seus diretórios de classe:
try:
    # Crie um dataset temporário para obter o número de classes, como você tinha antes
    temp_dataset = datasets.ImageFolder(root=DATASET_ROOT, transform=transforms.ToTensor())
    num_classes = len(temp_dataset.classes)
    # É crucial que `class_names` corresponda à ordem das classes que o modelo InceptionV3 aprendeu.
    # Se você carregou `best_inceptionv3_model_multiclass.h5` que foi treinado com `test_dataset.classes`,
    # então `temp_dataset.classes` deve ser a mesma ordem.
    # Ajuste `class_names` para refletir `temp_dataset.classes` se houver dúvida.
    class_names = temp_dataset.classes # Use as classes inferidas do dataset para garantir correspondência
    print(f"Classes carregadas do dataset: {class_names}")

except Exception as e:
    print(f"Erro ao inferir classes do DATASET_ROOT: {e}. Usando class_names predefinido.")
    # Se o carregamento do dataset falhar, use o num_classes baseado no seu class_names predefinido
    num_classes = len(class_names)


model = Model(num_classes=num_classes).to(DEVICE)
try:
    # Certifique-se de que 'best_inceptionv3_model_multiclass.h5' está no caminho correto
    model.load_state_dict(torch.load('best_inceptionv3_model_multiclass.h5', map_location=DEVICE))
    model.eval()
    print("Modelo carregado com sucesso.")
except FileNotFoundError:
    print(f"Erro: O arquivo do modelo 'best_inceptionv3_model_multiclass.h5' não foi encontrado. Verifique o caminho.")
    exit() # Aborta se o modelo não puder ser carregado
except Exception as e:
    print(f"Erro ao carregar o modelo: {e}")
    exit() # Aborta em caso de outros erros de carregamento


def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize((299, 299)), # Mantendo 299x299, mas considere 224x224 para economia de memória
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(DEVICE)
    # raw_img deve ser np.float32 e entre 0-1
    np_img = np.array(image.resize((299, 299))) / 255.0
    return tensor, np_img

def make_gradcam(model, img_tensor, target_layer='Mixed_7c'):
    gradients, activations = [], []

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0])

    def forward_hook(module, input, output):
        activations.append(output)

    # Verifica se o modelo é o wrapper `Model` ou o `inception_model` diretamente
    if hasattr(model, 'inception_model'):
        target_module = dict(model.inception_model.named_modules())[target_layer]
    else:
        target_module = dict(model.named_modules())[target_layer]

    fh = target_module.register_forward_hook(forward_hook)
    bh = target_module.register_full_backward_hook(backward_hook)

    img_tensor.requires_grad_(True)
    output = model(img_tensor)
    class_idx = torch.argmax(output).item() # Use .item() para obter o valor escalar
    loss = output[0, class_idx]
    model.zero_grad()
    loss.backward()

    grad = gradients[0]
    act = activations[0]
    pooled_grad = torch.mean(grad, dim=[0, 2, 3])
    for i in range(act.shape[1]):
        act[:, i, :, :] *= pooled_grad[i]
    heatmap = act.mean(dim=1).squeeze().detach().cpu().numpy()
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (heatmap.max() + 1e-12)

    fh.remove()
    bh.remove()
    return heatmap

def get_saliency_map(method, model, input_tensor, target_idx):
    input_tensor = input_tensor.clone().detach().requires_grad_(True)

    if method == "Saliency":
        explainer = Saliency(model)
        attributions = explainer.attribute(input_tensor, target=target_idx)
    elif method == "InputXGradient":
        explainer = InputXGradient(model)
        attributions = explainer.attribute(input_tensor, target=target_idx)
    elif method == "IntegratedGradients":
        explainer = IntegratedGradients(model)
        # Reduzindo n_steps e internal_batch_size para economizar memória
        attributions = explainer.attribute(input_tensor, target=target_idx, n_steps=5, internal_batch_size=1)
    elif method == "SmoothGrad":
        saliency = Saliency(model)
        explainer = NoiseTunnel(saliency)
        # Reduzindo nt_samples para economizar memória
        attributions = explainer.attribute(input_tensor, nt_type='smoothgrad', nt_samples=5, target=target_idx)
    elif method == "GradCAM":
        cam = make_gradcam(model, input_tensor)
        # Certifique-se de que o redimensionamento é apropriado para a imagem de entrada
        cam_resized = resize(cam, (299, 299), mode='reflect', anti_aliasing=True) # Redimensiona para o tamanho de entrada do modelo
        return cam_resized
    else:
        raise ValueError(f"Método {method} desconhecido.")

    attr_np = attributions.detach().cpu().numpy().squeeze()
    # Para saliency maps 2D, a soma sobre o canal de cor não é necessária se já é 2D
    if attr_np.ndim == 3 and attr_np.shape[0] in [1, 3]: # Se ainda tiver canais (ex: RGB)
        attr_np = np.abs(attr_np).sum(axis=0)
    elif attr_np.ndim == 2: # Se já for 2D
        attr_np = np.abs(attr_np) # Apenas pega o valor absoluto

    attr_np = (attr_np - attr_np.min()) / (attr_np.max() - attr_np.min() + 1e-12)
    return attr_np

def salience_entropy(smap):
    # Garante que smap é non-negative e some para 1 antes de calcular a entropia
    smap = smap / (np.sum(smap) + 1e-12)
    # Adiciona um pequeno valor para evitar log(0)
    return entropy(smap.flatten() + 1e-12)

def reaction_to_noise(model, img_tensor, salmap):
    noisy = img_tensor + torch.normal(0, 0.05, img_tensor.shape).to(DEVICE)
    # É importante usar `model.eval()` e `with torch.no_grad()` para previsões,
    # mas o Captum precisa de gradientes habilitados para atribuições.
    # Neste caso, para obter `target_idx` e `noisy_salmap`, o `get_saliency_map`
    # já lida com `requires_grad_`.
    # Apenas certifique-se de que o `model` esteja em modo de avaliação se não for para treinamento.

    with torch.no_grad(): # Para a previsão da classe, não precisamos de gradientes
        output = model(noisy)
        target_idx = torch.argmax(output).item()

    # Passa o modelo e o tensor (que será clonado e terá gradientes habilitados dentro de get_saliency_map)
    noisy_salmap = get_saliency_map("GradCAM", model, noisy, target_idx)
    
    noisy_salmap_resized = resize(noisy_salmap, salmap.shape, mode='reflect', anti_aliasing=True)
    
    return np.mean(np.abs(salmap - noisy_salmap_resized))

def geometrical_robustness(model, raw_img, salmap):
    # raw_img é um numpy array (H, W, C) float32 entre 0-1
    flipped = np.flip(raw_img, axis=1).copy()  # flip horizontal
    
    # Transforma o numpy array para tensor para o modelo
    # Certifique-se de que a ordem dos canais está correta para ToTensor() (C, H, W)
    # E normalize como a imagem original
    transform_for_flipped = transforms.Compose([
        transforms.ToPILImage(), # Converte para PIL para aplicar as mesmas transformações
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # Convertendo numpy array para PIL Image antes de aplicar transforms
    flipped_tensor = transform_for_flipped((flipped * 255).astype(np.uint8)).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        output = model(flipped_tensor)
        target_idx = torch.argmax(output).item()

    # A função get_saliency_map já cuidará de requires_grad_ para o input_tensor
    flipped_salmap = get_saliency_map("GradCAM", model, flipped_tensor, target_idx)

    # Redimensiona para o shape original do mapa
    flipped_salmap_resized = resize(flipped_salmap, salmap.shape, mode='reflect', anti_aliasing=True)

    return np.mean(np.abs(salmap - flipped_salmap_resized))

def accuracy_over_segmentation(model, image, saliency_map, segments=50):
    # A imagem de entrada para slic deve ser (H, W, C) e uint8
    salmap_resized = resize(saliency_map, (image.shape[0], image.shape[1]), mode='reflect', anti_aliasing=True)
    img_uint8 = (image * 255).astype(np.uint8) # Garante que a imagem esteja em uint8 para SLIC
    
    # O SLIC espera uma imagem colorida (H, W, 3) ou escala de cinza (H, W)
    # Se img_uint8 tiver 4 canais (RGBA), converta para RGB
    if img_uint8.shape[-1] == 4:
        img_uint8 = img_uint8[:, :, :3]

    segs = slic(img_uint8, n_segments=segments, compactness=20, enforce_connectivity=True, start_label=0)
    
    # Certifica-se de que salmap_resized e segs têm as mesmas dimensões espaciais
    if salmap_resized.shape != segs.shape:
        # Se o salmap_resized não for 2D, converta-o para 2D (média nos canais, por exemplo)
        if salmap_resized.ndim == 3:
            salmap_resized = np.mean(salmap_resized, axis=-1)
        salmap_resized = resize(salmap_resized, (segs.shape[0], segs.shape[1]), mode='reflect', anti_aliasing=True)
        
    relevance = np.array([np.mean(salmap_resized[segs == i]) for i in range(np.max(segs) + 1)])
    top_k = np.argsort(relevance)[-int(segments * 0.2):]
    
    mask = np.isin(segs, top_k).astype(np.float32)
    if mask.ndim == 2:
        mask = np.repeat(mask[:, :, np.newaxis], 3, axis=2) # Converte a máscara para 3 canais

    degraded = image * (1 - mask) # image já é float32 0-1, degraded também será
    
    # Para passar para o modelo, precisa ser tensor, C, H, W, normalizado
    transform_for_model = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((299, 299)), # Redimensiona para o tamanho de entrada do modelo
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    with torch.no_grad():
        # A entrada para o modelo deve ser um tensor [1, C, H, W]
        orig_pred = model(transform_for_model((image * 255).astype(np.uint8)).unsqueeze(0).to(DEVICE))[0]
        degr_pred = model(transform_for_model((degraded * 255).astype(np.uint8)).unsqueeze(0).to(DEVICE))[0]
    
    return np.abs(orig_pred.cpu().numpy() - degr_pred.cpu().numpy()).mean(), segs


# Coleta de arquivos de imagem
image_files = []
image_labels = []

# É crucial que `class_names` reflita exatamente as pastas no seu DATASET_ROOT
# e que também reflita as classes que o seu modelo foi treinado para prever.
for folder in class_names:
    folder_path = os.path.join(DATASET_ROOT, folder)
    if not os.path.exists(folder_path):
        print(f"Aviso: Diretório '{folder_path}' não encontrado. Verifique 'class_names' e 'DATASET_ROOT'.")
        continue # Pula para a próxima pasta se não existir

    files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]) # Adicionado .jpeg
    if not files:
        print(f"Aviso: Nenhum arquivo de imagem encontrado em '{folder_path}'.")
        continue

    for f in files:
        image_files.append(os.path.join(folder_path, f))
        image_labels.append(folder)

if not image_files:
    print("Nenhuma imagem encontrada nos diretórios especificados. Processamento abortado.")
    exit()

print(f"Total de {len(image_files)} imagens encontradas para processamento.")

resultados = []
n_rows = len(image_files)
# n_cols não é mais para plotagem no loop, então não é tão crítico, mas pode ser mantido
# se for para propósito de estimar o progresso ou dimensões lógicas
# fig, axs = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows)) # Comentado, pois pode consumir muita memória

for row, (path, label) in enumerate(zip(image_files, image_labels)):
    try:
        img_tensor, raw_img = preprocess_image(path)
        
        # Garante que o modelo esteja em modo de avaliação
        model.eval()
        with torch.no_grad():
            output = model(img_tensor)
            pred_idx = torch.argmax(output, dim=1).item()

        for col, method in enumerate(explic_methods): # col não é mais usado para subplot
            # Libera memória antes de cada cálculo de saliency map para evitar acúmulo
            torch.cuda.empty_cache()
            gc.collect()

            salmap = get_saliency_map(method, model, img_tensor.clone(), pred_idx) # Clona o tensor para evitar modificações in-place

            # Libera memória após o cálculo do salmap
            torch.cuda.empty_cache()
            gc.collect()

            entropy_val = salience_entropy(salmap)
            noise_val = reaction_to_noise(model, img_tensor.clone(), salmap) # Clona para outras funções
            geo_val = geometrical_robustness(model, raw_img.copy(), salmap) # Copia para evitar modificações in-place
            aos_val, _ = accuracy_over_segmentation(model, raw_img.copy(), salmap) # Copia para evitar modificações in-place

            resultado = {
                "imagem": os.path.basename(path),
                "classe_real": label,
                "classe_predita": class_names[pred_idx],
                "metodo_explicabilidade": method,
                "entropia": entropy_val,
                "ruido": noise_val,
                "geo": geo_val,
                "aos": aos_val
            }
            resultados.append(resultado)

            print(f"[{row+1}/{n_rows}] Imagem: {os.path.basename(path)} | Classe Real: {label} | Predita: {class_names[pred_idx]} | Técnica: {method} | Entropia: {entropy_val:.4f} | Ruído: {noise_val:.4f} | Geo: {geo_val:.4f} | AOS: {aos_val:.4f}")
            
    except Exception as e:
        print(f"Erro ao processar imagem {path} com a técnica {method}: {e}")
        # Opcional: Adicionar um resultado com erro para registro
        resultados.append({
            "imagem": os.path.basename(path),
            "classe_real": label,
            "classe_predita": "ERRO",
            "metodo_explicabilidade": method,
            "entropia": np.nan, "ruido": np.nan, "geo": np.nan, "aos": np.nan
        })
    
    # Libera memória no final de cada iteração de imagem
    del img_tensor, raw_img, output
    torch.cuda.empty_cache()
    gc.collect()


# plt.tight_layout() # Comentado
# plt.savefig("grid_explicabilidade_todas_imagens.png", dpi=300) # Comentado
# print("Grid salvo como grid_explicabilidade_todas_imagens.png") # Comentado

df_resultados = pd.DataFrame(resultados)
df_resultados.to_excel("resultados_explicabilidade_todas_imagens_inception.xlsx", index=False)
print("Planilha salva como resultados_explicabilidade_todas_imagens_inception.xlsx")